# `RecordManager: ABC`

Abstract interface used by the indexing API to track document keys, source groups, and write timestamps.

Timestamps must be monotonically increasing and should come from the storage server to reduce cleanup-related data-loss risk.

## Fields

```python
namespace: str # Namespace used to isolate the managed records
```

## Constructor

```python
RecordManager(
    namespace: str, # Namespace used by the record manager
) -> None
```

## Required subclass hooks

A concrete subclass must implement every method below. This class provides no optional hooks or asynchronous wrappers, and its abstract methods do not explicitly raise `NotImplementedError`.

### `create_schema`

Creates the record-manager storage schema.

```python
@abstractmethod
create_schema(
    self,
) -> None
```

### `acreate_schema`

Asynchronously creates the record-manager storage schema.

```python
@abstractmethod
async acreate_schema(
    self,
) -> None
```

### `get_time`

Returns the current server time as a high-resolution timestamp.

```python
@abstractmethod
get_time(
    self,
) -> float # Current server timestamp
```

### `aget_time`

Asynchronously returns the current server time as a high-resolution timestamp.

```python
@abstractmethod
async aget_time(
    self,
) -> float # Current server timestamp
```

### `update`

Upserts record keys, optional group IDs, and their update timestamps.

```python
@abstractmethod
update(
    self,
    keys: Sequence[str], # Record keys to upsert
    *,
    group_ids: Sequence[str | None] | None = None, # Group IDs corresponding to the keys
    time_at_least: float | None = None, # Optional lower bound for the storage timestamp
) -> None
```

Implementations should raise `ValueError` when the lengths of `keys` and `group_ids` do not match.

### `aupdate`

Asynchronously upserts record keys, optional group IDs, and their update timestamps.

```python
@abstractmethod
async aupdate(
    self,
    keys: Sequence[str], # Record keys to upsert
    *,
    group_ids: Sequence[str | None] | None = None, # Group IDs corresponding to the keys
    time_at_least: float | None = None, # Optional lower bound for the storage timestamp
) -> None
```

Implementations should raise `ValueError` when the lengths of `keys` and `group_ids` do not match.

### `exists`

Checks whether each supplied record key exists.

```python
@abstractmethod
exists(
    self,
    keys: Sequence[str], # Record keys to check
) -> list[bool] # Existence result corresponding to each key
```

### `aexists`

Asynchronously checks whether each supplied record key exists.

```python
@abstractmethod
async aexists(
    self,
    keys: Sequence[str], # Record keys to check
) -> list[bool] # Existence result corresponding to each key
```

### `list_keys`

Lists record keys matching timestamp and group filters.

```python
@abstractmethod
list_keys(
    self,
    *,
    before: float | None = None, # Include records updated before this timestamp
    after: float | None = None, # Include records updated after this timestamp
    group_ids: Sequence[str] | None = None, # Include records belonging to these groups
    limit: int | None = None, # Maximum number of keys to return
) -> list[str] # Matching record keys
```

### `alist_keys`

Asynchronously lists record keys matching timestamp and group filters.

```python
@abstractmethod
async alist_keys(
    self,
    *,
    before: float | None = None, # Include records updated before this timestamp
    after: float | None = None, # Include records updated after this timestamp
    group_ids: Sequence[str] | None = None, # Include records belonging to these groups
    limit: int | None = None, # Maximum number of keys to return
) -> list[str] # Matching record keys
```

### `delete_keys`

Deletes the supplied record keys.

```python
@abstractmethod
delete_keys(
    self,
    keys: Sequence[str], # Record keys to delete
) -> None
```

### `adelete_keys`

Asynchronously deletes the supplied record keys.

```python
@abstractmethod
async adelete_keys(
    self,
    keys: Sequence[str], # Record keys to delete
) -> None
```



In [ ]:
import time # Import time for timestamps
from collections.abc import Sequence # Import Sequence for type annotations
from langchain_core.indexing.base import RecordManager # Import the abstract RecordManager class


class SimpleRecordManager(RecordManager): # Create a concrete RecordManager implementation
    def __init__(self, namespace: str) -> None: # Initialize the record manager
        super().__init__(namespace) # Initialize the parent class
        self.records: dict[str, tuple[str | None, float]] = {} # Store group IDs and timestamps

    def create_schema(self) -> None: # Create the storage schema
        print("Schema created.") # Display a confirmation message

    async def acreate_schema(self) -> None: # Create the schema asynchronously
        self.create_schema() # Reuse the synchronous method

    def get_time(self) -> float: # Return the current timestamp
        return time.time() # Return the current system time

    async def aget_time(self) -> float: # Return the timestamp asynchronously
        return self.get_time() # Reuse the synchronous method

    def update(
        self,
        keys: Sequence[str], # Keys to add or update
        *,
        group_ids: Sequence[str | None] | None = None, # Optional group IDs
        time_at_least: float | None = None, # Optional minimum timestamp
    ) -> None:
        if group_ids is not None and len(keys) != len(group_ids): # Check whether lengths match
            raise ValueError("keys and group_ids must have the same length") # Raise an error for invalid input

        timestamp = self.get_time() # Get the current timestamp

        if time_at_least is not None and timestamp < time_at_least: # Check the minimum timestamp
            raise ValueError("Current time is earlier than time_at_least") # Raise an error for invalid time

        groups = group_ids if group_ids is not None else [None] * len(keys) # Create default groups

        for key, group_id in zip(keys, groups): # Process each key and group
            self.records[key] = (group_id, timestamp) # Add or replace the record

    async def aupdate(
        self,
        keys: Sequence[str], # Keys to add or update
        *,
        group_ids: Sequence[str | None] | None = None, # Optional group IDs
        time_at_least: float | None = None, # Optional minimum timestamp
    ) -> None:
        self.update( # Reuse the synchronous update method
            keys,
            group_ids=group_ids,
            time_at_least=time_at_least,
        )

    def exists(self, keys: Sequence[str]) -> list[bool]: # Check whether keys exist
        return [key in self.records for key in keys] # Return one Boolean for each key

    async def aexists(self, keys: Sequence[str]) -> list[bool]: # Check keys asynchronously
        return self.exists(keys) # Reuse the synchronous method

    def list_keys(
        self,
        *,
        before: float | None = None, # Include records before this timestamp
        after: float | None = None, # Include records after this timestamp
        group_ids: Sequence[str] | None = None, # Include records from these groups
        limit: int | None = None, # Limit the number of returned keys
    ) -> list[str]:
        result: list[str] = [] # Store matching keys

        for key, (group_id, timestamp) in self.records.items(): # Check every stored record
            if before is not None and timestamp >= before: # Apply the before filter
                continue # Skip the current record

            if after is not None and timestamp <= after: # Apply the after filter
                continue # Skip the current record

            if group_ids is not None and group_id not in group_ids: # Apply the group filter
                continue # Skip the current record

            result.append(key) # Add the matching key

            if limit is not None and len(result) >= limit: # Check whether the limit is reached
                break # Stop collecting keys

        return result # Return matching keys

    async def alist_keys(
        self,
        *,
        before: float | None = None, # Include records before this timestamp
        after: float | None = None, # Include records after this timestamp
        group_ids: Sequence[str] | None = None, # Include records from these groups
        limit: int | None = None, # Limit the number of returned keys
    ) -> list[str]:
        return self.list_keys( # Reuse the synchronous list_keys method
            before=before,
            after=after,
            group_ids=group_ids,
            limit=limit,
        )

    def delete_keys(self, keys: Sequence[str]) -> None: # Delete stored records
        for key in keys: # Process every requested key
            self.records.pop(key, None) # Delete the key when it exists

    async def adelete_keys(self, keys: Sequence[str]) -> None: # Delete records asynchronously
        self.delete_keys(keys) # Reuse the synchronous method


manager = SimpleRecordManager("demo") # Create the record manager

await manager.acreate_schema() # Create the schema

await manager.aupdate( # Add two records
    ["doc1", "doc2"],
    group_ids=["python", "langchain"],
)

print("Current time:", await manager.aget_time()) # Display the current timestamp
print("Exists:", await manager.aexists(["doc1", "doc3"])) # Check existing and missing keys
print("All keys:", await manager.alist_keys()) # Display all stored keys
print("Python group:", await manager.alist_keys(group_ids=["python"])) # Filter keys by group
print("Limited keys:", await manager.alist_keys(limit=1)) # Return only one key

await manager.adelete_keys(["doc1"]) # Delete the first record

print("After deletion:", await manager.alist_keys()) # Display the remaining keys

# `InMemoryRecordManager: RecordManager`

In-memory `RecordManager` implementation intended for testing.

## Fields

```python
namespace: str # Namespace used by the record manager
records: dict[str, _Record] # Stored group IDs and update timestamps keyed by record key
```

## Constructor

```python
InMemoryRecordManager(
    namespace: str, # Namespace used by the record manager
) -> None
```

## Methods

### `create_schema`

Performs no work because the in-memory structure is initialized by the constructor.

```python
create_schema(
    self,
) -> None
```

### `acreate_schema`

Asynchronous no-op schema creation.

```python
async acreate_schema(
    self,
) -> None
```

### `get_time`

Returns the local system time from `time.time()`.

```python
@override
get_time(
    self,
) -> float # Current local timestamp
```

### `aget_time`

Asynchronously delegates to `get_time()`.

```python
@override
async aget_time(
    self,
) -> float # Current local timestamp
```

### `update`

Stores or replaces records with the current timestamp.

```python
update(
    self,
    keys: Sequence[str], # Record keys to upsert
    *,
    group_ids: Sequence[str | None] | None = None, # Group IDs corresponding to the keys
    time_at_least: float | None = None, # Timestamp that must not be in the future
) -> None
```

Raises `ValueError` when a non-empty `group_ids` sequence has a different length from `keys`, or when `time_at_least` is later than the current time.

### `aupdate`

Asynchronously delegates to `update()`.

```python
async aupdate(
    self,
    keys: Sequence[str], # Record keys to upsert
    *,
    group_ids: Sequence[str | None] | None = None, # Group IDs corresponding to the keys
    time_at_least: float | None = None, # Timestamp that must not be in the future
) -> None
```

### `exists`

Returns one Boolean result for each supplied key.

```python
exists(
    self,
    keys: Sequence[str], # Record keys to check
) -> list[bool] # Whether each key is present
```

### `aexists`

Asynchronously delegates to `exists()`.

```python
async aexists(
    self,
    keys: Sequence[str], # Record keys to check
) -> list[bool] # Whether each key is present
```

### `list_keys`

Returns stored keys that satisfy the supplied filters.

```python
list_keys(
    self,
    *,
    before: float | None = None, # Include records with timestamps earlier than this value
    after: float | None = None, # Include records with timestamps later than this value
    group_ids: Sequence[str] | None = None, # Include records belonging to these groups
    limit: int | None = None, # Maximum number of keys to return
) -> list[str] # Matching keys
```

The timestamp boundaries are exclusive.

### `alist_keys`

Asynchronously delegates to `list_keys()`.

```python
async alist_keys(
    self,
    *,
    before: float | None = None, # Include records with timestamps earlier than this value
    after: float | None = None, # Include records with timestamps later than this value
    group_ids: Sequence[str] | None = None, # Include records belonging to these groups
    limit: int | None = None, # Maximum number of keys to return
) -> list[str] # Matching keys
```

### `delete_keys`

Deletes existing keys and ignores keys that are not present.

```python
delete_keys(
    self,
    keys: Sequence[str], # Record keys to delete
) -> None
```

### `adelete_keys`

Asynchronously delegates to `delete_keys()`.

```python
async adelete_keys(
    self,
    keys: Sequence[str], # Record keys to delete
) -> None
```

In [ ]:
from langchain_core.indexing.base import InMemoryRecordManager # Import the in-memory record manager

manager = InMemoryRecordManager(namespace="demo") # Create a record manager for the demo namespace

manager.create_schema() # Initialize storage; this method performs no action for in-memory storage

manager.update( # Add three records
    ["doc1", "doc2", "doc3"], # Provide record keys
    group_ids=["python", "langchain", "python"], # Assign each record to a group
)

print("Current time:", manager.get_time()) # Display the current timestamp

result = manager.exists(["doc1", "doc4"]) # Check an existing and a missing key
print("Existence result:", result) # Display [True, False]

all_keys = manager.list_keys() # Retrieve all stored keys
print("All keys:", all_keys) # Display all keys

python_keys = manager.list_keys(group_ids=["python"]) # Retrieve records from the python group
print("Python group:", python_keys) # Display doc1 and doc3

limited_keys = manager.list_keys(limit=2) # Retrieve a maximum of two keys
print("Limited keys:", limited_keys) # Display the first two keys

manager.delete_keys(["doc2", "missing"]) # Delete doc2 and ignore the missing key
print("After deletion:", manager.list_keys()) # Display the remaining keys

await manager.acreate_schema() # Call the asynchronous schema method

await manager.aupdate( # Add another record asynchronously
    ["doc4"], # Provide the new key
    group_ids=["langchain"], # Assign its group
)

async_exists = await manager.aexists(["doc3", "doc4"]) # Check keys asynchronously
print("Async existence:", async_exists) # Display [True, True]

async_keys = await manager.alist_keys(group_ids=["langchain"]) # List a group asynchronously
print("Async LangChain group:", async_keys) # Display doc4

print("Async current time:", await manager.aget_time()) # Get time asynchronously

await manager.adelete_keys(["doc1", "doc4"]) # Delete records asynchronously
print("Final keys:", await manager.alist_keys()) # Display the final stored keys

# `UpsertResponse: TypedDict`

Response returned by an ID-based upsert operation.

```python
succeeded: list[str] # IDs successfully added or updated
failed: list[str] # IDs that failed to be added or updated
```

When no failures occur, `failed` is empty and `succeeded` follows the input-document order. When generated IDs and partial failures are combined, the correspondence between inputs and generated IDs is not defined.



In [ ]:
from langchain_core.indexing.base import UpsertResponse # Import the response type


def upsert_documents(document_ids: list[str]) -> UpsertResponse: # Simulate an upsert operation
    succeeded: list[str] = [] # Store successfully processed IDs
    failed: list[str] = [] # Store failed IDs

    for document_id in document_ids: # Process every document ID
        if document_id == "doc2": # Simulate a failure for doc2
            failed.append(document_id) # Add the ID to the failed list
        else:
            succeeded.append(document_id) # Add the ID to the succeeded list

    return { # Return the UpsertResponse dictionary
        "succeeded": succeeded, # Include successful IDs
        "failed": failed, # Include failed IDs
    }


response = upsert_documents(["doc1", "doc2", "doc3"]) # Perform the simulated upsert

print("Complete response:", response) # Display the complete response
print("Succeeded IDs:", response["succeeded"]) # Display successfully upserted IDs
print("Failed IDs:", response["failed"]) # Display failed IDs

# `DeleteResponse: TypedDict`

Optional details returned by a delete operation. The declaration uses `total=False`, so every field is optional.

```python
num_deleted: int # Number of items actually deleted
succeeded: Sequence[str] # IDs that were actually deleted
failed: Sequence[str] # IDs whose deletion failed
num_failed: int # Number of failed deletions
```

Deleting an ID that does not exist is not considered a failure and should not be counted as an actual deletion.

In [ ]:
from langchain_core.indexing.base import DeleteResponse # Import the response type


def delete_documents( # Define a simulated delete function
    requested_ids: list[str], # Receive document IDs to delete
    existing_ids: set[str], # Receive currently stored document IDs
) -> DeleteResponse:
    succeeded: list[str] = [] # Store IDs that were actually deleted
    failed: list[str] = [] # Store IDs whose deletion failed

    for document_id in requested_ids: # Process every requested ID
        if document_id == "doc3": # Simulate a deletion failure
            failed.append(document_id) # Record the failed ID
        elif document_id in existing_ids: # Check whether the document exists
            existing_ids.remove(document_id) # Delete the existing document
            succeeded.append(document_id) # Record the successfully deleted ID
        else:
            pass # Ignore missing IDs because they are not failures

    response: DeleteResponse = { # Create the deletion response
        "num_deleted": len(succeeded), # Count actual deletions
        "succeeded": succeeded, # Include successfully deleted IDs
        "failed": failed, # Include failed deletion IDs
        "num_failed": len(failed), # Count failed deletions
    }

    return response # Return the deletion result


stored_documents = {"doc1", "doc2", "doc3"} # Create existing document IDs

result = delete_documents( # Run the simulated deletion
    ["doc1", "doc3", "doc5"], # doc1 exists, doc3 fails, and doc5 is missing
    stored_documents, # Provide the stored IDs
)

print("Delete response:", result) # Display the complete response
print("Remaining documents:", stored_documents) # Display documents still stored

# `DocumentIndex: BaseRetriever`

Abstract beta interface for indexing, retrieving, and searching documents with IDs and metadata.

Marked with `@beta(message="Added in 0.2.29. The abstraction is subject to change.")`.

## Required subclass hooks

This module requires concrete subclasses to implement `upsert()`, `delete()`, and `get()`. Their asynchronous counterparts are provided as executor wrappers and do not raise `NotImplementedError` by default.

### `upsert`

Adds new documents or updates existing documents using their IDs when available.

```python
@abc.abstractmethod
upsert(
    self,
    items: Sequence[Document], # Documents to add or update
    /,
    **kwargs: Any, # Implementation-specific options
) -> UpsertResponse # Successful and failed document IDs
```

An implementation may generate IDs for documents without IDs. Explicit IDs are recommended when partial failures must be associated with their inputs.

### `delete`

Deletes documents by ID or by implementation-specific criteria.

```python
@abc.abstractmethod
delete(
    self,
    ids: list[str] | None = None, # Document IDs to delete
    **kwargs: Any, # Implementation-specific deletion options
) -> DeleteResponse # Optional deletion details
```

An implementation should raise `ValueError` when called without any deletion criteria.

### `get`

Retrieves documents by ID.

```python
@abc.abstractmethod
get(
    self,
    ids: Sequence[str], # Document IDs to retrieve
    /,
    **kwargs: Any, # Implementation-specific retrieval options
) -> list[Document] # Documents that were found
```

The result may contain fewer documents than requested. Implementations should not raise because an ID is missing, and callers must not assume the returned order matches the input order.

## Methods

### `aupsert`

Runs `upsert()` asynchronously through `run_in_executor()`.

```python
async aupsert(
    self,
    items: Sequence[Document], # Documents to add or update
    /,
    **kwargs: Any, # Options forwarded to upsert()
) -> UpsertResponse # Successful and failed document IDs
```

### `adelete`

Runs `delete()` asynchronously through `run_in_executor()`.

```python
async adelete(
    self,
    ids: list[str] | None = None, # Document IDs to delete
    **kwargs: Any, # Options forwarded to delete()
) -> DeleteResponse # Optional deletion details
```

Calling it without deletion criteria relies on `delete()` to raise `ValueError`.

### `aget`

Runs `get()` asynchronously through `run_in_executor()`.

```python
async aget(
    self,
    ids: Sequence[str], # Document IDs to retrieve
    /,
    **kwargs: Any, # Options forwarded to get()
) -> list[Document] # Documents that were found
```

In [ ]:
from langchain_core.documents import Document # Import the Document class
from langchain_core.indexing.in_memory import InMemoryDocumentIndex # Import a concrete DocumentIndex

index = InMemoryDocumentIndex(top_k=2) # Create an in-memory document index

documents = [ # Create documents with explicit IDs
    Document(id="doc1", page_content="Python is a programming language."), # Create the first document
    Document(id="doc2", page_content="LangChain helps build AI applications."), # Create the second document
    Document(id="doc3", page_content="Python is commonly used for AI."), # Create the third document
] # Finish the document list

upsert_result = index.upsert(documents) # Add the documents to the index

print("Upsert result:", upsert_result) # Display successful and failed IDs

found_documents = index.get(["doc1", "doc3", "missing"]) # Retrieve documents by ID

for document in found_documents: # Visit each retrieved document
    print(document.id, ":", document.page_content) # Display its ID and content

search_results = index.invoke("Python") # Search for documents containing Python

print("\nSearch results:") # Display a heading

for document in search_results: # Visit each search result
    print(document.id, ":", document.page_content) # Display the matching document

delete_result = index.delete(["doc2", "missing"]) # Delete an existing and a missing ID

print("\nDelete result:", delete_result) # Display the deletion details
print("Remaining IDs:", list(index.store)) # Display the remaining document IDs

async_upsert_result = await index.aupsert( # Add a document asynchronously in Jupyter
    [Document(id="doc4", page_content="Async methods run using an executor.")]
) # Finish the asynchronous upsert

print("\nAsync upsert:", async_upsert_result) # Display the asynchronous upsert result

async_documents = await index.aget(["doc4"]) # Retrieve the document asynchronously

print("Async get:", async_documents[0].page_content) # Display the retrieved content

async_delete_result = await index.adelete(["doc4"]) # Delete the document asynchronously

print("Async delete:", async_delete_result) # Display the asynchronous deletion result